# 03 — Beitragsextraktion (SHAP / EBM)

Extrahiert globale Feature-Importance und lokale Beiträge für die **5 fixen Testinstanzen**
(`INSTANCE_IDS = [42, 100, 250, 500, 1337]`) für XGBoost und EBM mit **Option 2 (Poisson-Log)**.

- **XGBoost**: SHAP TreeExplainer — Beiträge im Log-Raum (additiv)
- **EBM**: interpret `explain_local` — Terme ebenfalls im Log-Raum

Alle Ausgaben werden als JSON in `explanations/` gespeichert und von Notebooks 04–06 gelesen.

In [1]:
from __future__ import annotations

import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd

from utils import INSTANCE_IDS, EXPLANATIONS_DIR, MODELS_DIR, RESULTS_DIR
from utils.data import load_train_test
from utils.explanations import build_global, build_local, save_explanation

LOSS_KEY = "poisson_log"

print(f"Testinstanzen: {INSTANCE_IDS}")
print(f"Loss-Option:   {LOSS_KEY}")
print(f"Ausgabe:       {EXPLANATIONS_DIR}")

Testinstanzen: [224, 580, 1041, 1481, 1677, 2058, 2510, 3543, 3847, 4454]
Loss-Option:   poisson_log
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/explanations


## 1 · Daten und Modelle laden

In [2]:
X_train, y_train, X_test, y_test = load_train_test()
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

xgb = joblib.load(MODELS_DIR / f"xgb_{LOSS_KEY}.pkl")
ebm = joblib.load(MODELS_DIR / f"ebm_{LOSS_KEY}.pkl")
print("Modelle geladen.")

metrics = json.loads((RESULTS_DIR / f"model_metrics_{LOSS_KEY}.json").read_text())["metrics"]
print("Metriken geladen:", list(metrics.keys()))

X_train: (12165, 9)  |  X_test: (5214, 9)
Modelle geladen.
Metriken geladen: ['xgb', 'ebm']


## 2 · Globale Erklärungen

In [3]:
print("Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...")
global_xgb = build_global(xgb, "xgb", X_train, metrics["xgb"])

path = save_explanation(global_xgb, f"global_xgb_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (XGBoost):")
for entry in global_xgb["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/explanations/global_xgb_poisson_log.json

Top-5 Features (XGBoost):
  # 1  hr            0.8424
  # 2  yr            0.2071
  # 3  temp          0.1900
  # 4  weekday       0.1612
  # 5  mnth          0.1262


In [4]:
print("Berechne globale EBM-Erklärung...")
global_ebm = build_global(ebm, "ebm", X_train, metrics["ebm"])

path = save_explanation(global_ebm, f"global_ebm_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (EBM):")
for entry in global_ebm["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale EBM-Erklärung...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation_1205/explanations/global_ebm_poisson_log.json

Top-5 Features (EBM):
  # 1  hr            0.9068
  # 2  temp          0.2364
  # 3  yr            0.2283
  # 4  mnth          0.0967
  # 5  hum           0.0731


## 3 · Lokale Erklärungen für die 5 Testinstanzen

In [5]:
for iid in INSTANCE_IDS:
    local = build_local(xgb, "xgb", X_test, y_test, iid)
    path = save_explanation(local, f"local_xgb_{LOSS_KEY}_inst{iid}.json")
    top = local["contributions"][0]
    print(f"  inst={iid:4d}  pred={local['prediction']:6.1f}  y_true={local['y_true']:6.0f}  "
          f"top_feature={top['feature']} ({top['contribution']:+.3f})  -> {path.name}")

  inst= 224  pred= 286.9  y_true=   270  top_feature=hr (+0.287)  -> local_xgb_poisson_log_inst224.json
  inst= 580  pred=  12.1  y_true=     5  top_feature=hr (-1.468)  -> local_xgb_poisson_log_inst580.json
  inst=1041  pred= 208.2  y_true=   229  top_feature=weekday (-0.173)  -> local_xgb_poisson_log_inst1041.json
  inst=1481  pred= 108.3  y_true=   113  top_feature=hr (-0.752)  -> local_xgb_poisson_log_inst1481.json
  inst=1677  pred= 180.3  y_true=   145  top_feature=yr (-0.260)  -> local_xgb_poisson_log_inst1677.json
  inst=2058  pred= 179.5  y_true=   238  top_feature=yr (-0.240)  -> local_xgb_poisson_log_inst2058.json
  inst=2510  pred= 368.9  y_true=   337  top_feature=hr (+0.558)  -> local_xgb_poisson_log_inst2510.json
  inst=3543  pred= 660.0  y_true=   691  top_feature=hr (+0.545)  -> local_xgb_poisson_log_inst3543.json
  inst=3847  pred= 141.4  y_true=   122  top_feature=hr (-0.764)  -> local_xgb_poisson_log_inst3847.json
  inst=4454  pred= 382.3  y_true=   311  top_feature

In [6]:
for iid in INSTANCE_IDS:
    local = build_local(ebm, "ebm", X_test, y_test, iid)
    path = save_explanation(local, f"local_ebm_{LOSS_KEY}_inst{iid}.json")
    top = local["contributions"][0]
    print(f"  inst={iid:4d}  pred={local['prediction']:6.1f}  y_true={local['y_true']:6.0f}  "
          f"top_feature={top['feature']} ({top['contribution']:+.3f})  -> {path.name}")

  inst= 224  pred= 290.1  y_true=   270  top_feature=hr (+0.582)  -> local_ebm_poisson_log_inst224.json
  inst= 580  pred=   9.7  y_true=     5  top_feature=hr (-0.779)  -> local_ebm_poisson_log_inst580.json
  inst=1041  pred= 221.3  y_true=   229  top_feature=hr (+0.475)  -> local_ebm_poisson_log_inst1041.json
  inst=1481  pred= 124.9  y_true=   113  top_feature=hr (-0.309)  -> local_ebm_poisson_log_inst1481.json
  inst=1677  pred= 178.5  y_true=   145  top_feature=hr (+0.663)  -> local_ebm_poisson_log_inst1677.json
  inst=2058  pred= 164.7  y_true=   238  top_feature=hr (+0.582)  -> local_ebm_poisson_log_inst2058.json
  inst=2510  pred= 373.4  y_true=   337  top_feature=hr (+0.866)  -> local_ebm_poisson_log_inst2510.json
  inst=3543  pred= 653.9  y_true=   691  top_feature=hr (+0.840)  -> local_ebm_poisson_log_inst3543.json
  inst=3847  pred= 137.1  y_true=   122  top_feature=hr (-0.364)  -> local_ebm_poisson_log_inst3847.json
  inst=4454  pred= 340.3  y_true=   311  top_feature=hr (

## 4 · Überblick gespeicherter Dateien

In [7]:
files = sorted(EXPLANATIONS_DIR.glob(f"*_{LOSS_KEY}*.json"))
print(f"Gespeicherte Erklärungsdateien ({len(files)}):")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:.1f} KB")

Gespeicherte Erklärungsdateien (22):
  global_ebm_poisson_log.json                    2.8 KB
  global_xgb_poisson_log.json                    2.8 KB
  local_ebm_poisson_log_inst1041.json            1.2 KB
  local_ebm_poisson_log_inst1481.json            1.2 KB
  local_ebm_poisson_log_inst1677.json            1.2 KB
  local_ebm_poisson_log_inst2058.json            1.2 KB
  local_ebm_poisson_log_inst224.json             1.2 KB
  local_ebm_poisson_log_inst2510.json            1.2 KB
  local_ebm_poisson_log_inst3543.json            1.2 KB
  local_ebm_poisson_log_inst3847.json            1.1 KB
  local_ebm_poisson_log_inst4454.json            1.2 KB
  local_ebm_poisson_log_inst580.json             1.2 KB
  local_xgb_poisson_log_inst1041.json            1.2 KB
  local_xgb_poisson_log_inst1481.json            1.2 KB
  local_xgb_poisson_log_inst1677.json            1.2 KB
  local_xgb_poisson_log_inst2058.json            1.2 KB
  local_xgb_poisson_log_inst224.json             1.2 KB
  local_xgb

## 5 · Sanity-Check: XGBoost vs. EBM — Top-Features im Vergleich

In [8]:
imp_xgb = {e["feature"]: e["rank"] for e in global_xgb["global_importance"]}
imp_ebm = {e["feature"]: e["rank"] for e in global_ebm["global_importance"]}

all_features = sorted(set(imp_xgb) | set(imp_ebm))
rows = [{"Feature": f, "Rank XGB": imp_xgb.get(f, "-"), "Rank EBM": imp_ebm.get(f, "-")} for f in all_features]
df = pd.DataFrame(rows).sort_values("Rank XGB")
print(df.to_string(index=False))

   Feature  Rank XGB  Rank EBM
        hr         1         1
        yr         2         3
      temp         3         2
   weekday         4         7
      mnth         5         4
       hum         6         5
weathersit         7         6
 windspeed         8         8
   holiday         9         9
